In [2]:
#импорт библиотек
#import библиотека как объект
import numpy as np # Работа с массивами и математическими операциями
import pandas as pd # Работа с таблицами и датасетами
import matplotlib.pyplot as plt # Построение графиков
from sklearn.model_selection import train_test_split # Разделение данных на обучающую и тестовую выборки
from sklearn.linear_model import LinearRegression # Линейная регрессия
from sklearn.preprocessing import PolynomialFeatures # Полиномиальные признаки
from sklearn.pipeline import Pipeline # Позволяет объединять несколько этапов обработки в одну модель
from sklearn.metrics import mean_squared_error, r2_score # Метрики качества регрессии

# #Для классификации
# from sklearn.tree import DecisionTreeClassifier
# from sklearn.metrics import accuracy_score, classification_report
# #Для кластеризации
# from sklearn.cluster import KMeans
# from sklearn.preprocessing import StandardScaler

In [ ]:
#Загрузка датасета
df = pd.read_csv("название.csv") #Загружаем датасет из файла CSV
#Если датасет уже есть в библиотеке:
# from sklearn.datasets import load_iris
# data = load_iris()
# X = data.data
# y = data.target
df

In [ ]:
#Выбор признаков X и целевой переменной y

#Получается X - все столбцы, кроме target, а y - только столбец target
#X → всё, по чему предсказываем
#y → то, что предсказываем
X = df.drop(columns=["target"])
y = df["target"]

#Если не нужны все столбцы:
# X = df[["признак1","признак2","признак3"]]
# y = df["target"]
df = df.dropna() # Удалить пропуски в данных
X = X.fillna(X.median(numeric_only=True)) # Заполнить пропуски медианой для числовых признаков
X = pd.get_dummies(X,columns=["тексовый_признак"],drop_first=True) # Преобразуем категориальные признаки в числовые с помощью one-hot encoding

In [ ]:
#Разделение данных
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42) # Разделяем данные на обучающую и тестовую выборки (80% обучение на 20% тестирование) 
#random_state=42 фиксируем случайное разделение, чтобы при каждом запуске получался одинаковый результат

#Линейная регрессия
model = LinearRegression() #model - ее название можно другое
model.fit(X_train, y_train) # Обучаем модель на обучающей выборке
#Предсказание на обучающей и тестовой выборках
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)
#Оценка регрессии с помощью метрик MSE и R²
train_mse = mean_squared_error(y_train, y_train_pred) #Меньше → лучше
test_mse = mean_squared_error(y_test, y_test_pred)
train_r2 = r2_score(y_train, y_train_pred) #Ближе к 1 → лучше
test_r2 = r2_score(y_test, y_test_pred)

print(f"Train MSE: {train_mse:.4f}") # 4 знака после запятой
print(f"Test MSE: {test_mse:.4f}")

print(f"Train R²: {train_r2:.4f}")
print(f"Test R²: {test_r2:.4f}")

#Поиск корня от среднеквадратичной ошибки (RMSE) для удобного чтения
train_rmse = np.sqrt(train_mse)
test_rmse = np.sqrt(test_mse)
print(f"Train RMSE: {train_rmse:.4f}")
print(f"Test RMSE: {test_rmse:.4f}")

In [ ]:
# Создаём полиномиальную модель 2-й степени. Она учитывает не только линейную зависимость, но и квадратичную.
poly_model = Pipeline([
    ("poly", PolynomialFeatures(degree=2, include_bias=False)), #include bias=False убирает константу, чтобы не было лишнего столбца. Короче надо
    ("linear", LinearRegression())
])
# Обучаем полиномиальную модель на обучающих данных
poly_model.fit(X_train, y_train)

# Получаем предсказания полиномиальной модели
y_train_pred_poly = poly_model.predict(X_train)
y_test_pred_poly = poly_model.predict(X_test)

# Считаем MSE полиномиальной модели
train_mse_poly = mean_squared_error(y_train, y_train_pred_poly)
test_mse_poly = mean_squared_error(y_test, y_test_pred_poly)

# Считаем R² полиномиальной модели
train_r2_poly = r2_score(y_train, y_train_pred_poly)
test_r2_poly = r2_score(y_test, y_test_pred_poly)

print("Полиномиальная модель:")
print(f"Train MSE: {train_mse_poly:.4f}")
print(f"Test MSE: {test_mse_poly:.4f}")
print(f"Train R²: {train_r2_poly:.4f}")
print(f"Test R²: {test_r2_poly:.4f}")

#если надо сравнить с метриками линейной модели, то можно вывести их вместе
# print("\nСравнение моделей:")
# print(f"Линейная  — Test MSE: {test_mse:.4f}, Test R²: {test_r2:.4f}")
# print(f"Полином 2 — Test MSE: {test_mse_poly:.4f}, Test R²: {test_r2_poly:.4f}")

In [ ]:
degrees = range(1, 5) # Степени полинома от 1 до 4
train_errors = [] #список для хранения ошибок на обучающей выборке
test_errors = [] #список для хранения ошибок на тестовой выборке

for degree in degrees: #пока степенеь в диапазоне степеней от 1 до 4, то есть 1,2,3,4 в нашем случае. Далее все как кобычно
    model_poly = Pipeline([
        ('poly', PolynomialFeatures(degree=degree, include_bias=False)),
        ('linear', LinearRegression())
    ])
    model_poly.fit(X_train, y_train)

    y_train_pred = model_poly.predict(X_train)
    y_test_pred = model_poly.predict(X_test)

    train_errors.append(mean_squared_error(y_train, y_train_pred))
    test_errors.append(mean_squared_error(y_test, y_test_pred))


for degree, train_mse, test_mse in zip(degrees, train_errors, test_errors): #Пока степенеь, ошибка на обучающей выборке и ошибка на тестовой выборке в одном списке, то выводим их
    print(f"Степень {degree}: train_MSE={train_mse:.4f}, test_MSE={test_mse:.4f}")


#Лучшая модель
best_index = test_errors.index(min(test_errors)) #Создаем переменную best_index, которая хранит индекс минимального значения в списке test_errors. 
#Это позволяет определить, какая степень полинома дала наименьшую ошибку на тестовой выборке.
best_degree = degrees[best_index] #Создаем переменную best_degree, которая хранит оптимальную степень полинома.
best_train_mse = train_errors[best_index]#Создаем переменную best_train_mse, которая хранит значение ошибки на обучающей выборке для оптимальной степени полинома.
best_test_mse = test_errors[best_index]#Создаем переменную best_test_mse, которая хранит значение ошибки на тестовой выборке для оптимальной степени полинома.

print(f"Оптимальная степень полинома: {best_degree}")
print(f"При ней: train_MSE = {best_train_mse:.4f} | test_MSE = {best_test_mse:.4f}")

# Построение графика тут вроде все понятно
plt.figure(figsize=(10, 6))
plt.plot(list(degrees), train_errors, marker='o', color='blue', label='Обучающая ошибка')
plt.plot(list(degrees), test_errors, marker='o', color='red', label='Тестовая ошибка')
plt.xlabel('Степень полинома')
plt.ylabel('MSE')
plt.title('Зависимость ошибки от степени полинома')
plt.xticks(list(degrees))# Указываем значения на оси X
plt.grid(True)
plt.legend()
plt.show()